# Data Preprocessing

In [0]:
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer

import GX_utils as U

spark = U.get_spark("GX_Preprocessing")
U.setup_storage(spark)
print("Databricks:", U.IS_DATABRICKS, "| raw data:", U.RAW_DIR)

## 1. Bronze: raw data with explicit schemas plus lineage columns

In [0]:
def add_lineage(df):
    return (df.withColumn("_ingested_at", F.current_timestamp())
              .withColumn("_source_file", F.col("_metadata.file_path")))

U.write_table(add_lineage(U.read_mental_health_raw(spark)), "bronze", "mental_health")
U.write_table(add_lineage(U.read_twitter_graph_raw(spark)), "bronze", "twitter_graph")

bronze_mental_health = U.read_table(spark, "bronze", "mental_health")
bronze_twitter_graph = U.read_table(spark, "bronze", "twitter_graph")
n_bronze_text, n_bronze_edges = bronze_mental_health.count(), bronze_twitter_graph.count()
print(f"bronze texts: {n_bronze_text:,} | bronze relationships: {n_bronze_edges:,}")

## 2. Silver text: deduplicate, clean, index labels in a fixed alphabetical order and keep style features from the raw text

In [0]:
valid_text = (bronze_mental_health
    .dropDuplicates(["ID"])
    .filter(F.col("Text").isNotNull() & F.col("Emotion").isin(U.EMOTION_LABELS))
    .withColumn("cleaned_text", U.clean_text(F.col("Text")))
    .filter(F.length("cleaned_text") > 2))

indexer_model = StringIndexer(inputCol="Emotion", outputCol="label", stringOrderType="alphabetAsc").fit(valid_text)
assert list(indexer_model.labels) == U.EMOTION_LABELS

letters = F.length(F.regexp_replace("Text", "[^A-Za-z]", ""))
uppers = F.length(F.regexp_replace("Text", "[^A-Z]", ""))

silver_mental_health = (indexer_model.transform(valid_text)
    .withColumn("word_count", F.size(F.split(F.trim("Text"), r"\s+")))
    .withColumn("char_length", F.length("Text"))
    .withColumn("upper_ratio", F.when(letters > 0, uppers / letters).otherwise(0.0))
    .withColumn("is_distress", F.col("Emotion").isin(U.DISTRESS_EMOTIONS).cast("int"))
    .select("ID", "Text", "cleaned_text", "Emotion", "label", "is_distress",
            "word_count", "char_length", "upper_ratio", "_ingested_at"))

U.write_table(silver_mental_health, "silver", "mental_health")
silver_mental_health = U.read_table(spark, "silver", "mental_health")
n_silver_text = silver_mental_health.count()
print(f"silver texts: {n_silver_text:,} (removed {n_bronze_text - n_silver_text:,})")
silver_mental_health.select("ID", "cleaned_text", "Emotion", "label", "word_count").show(5, truncate=60)

## 3. Silver graph: Source IDs were exported

In [0]:
ids = bronze_twitter_graph.select(
    "Source", "Destination",
    U.normalize_twitter_id(F.col("Source")).alias("src_key"),
    U.normalize_twitter_id(F.col("Destination")).alias("dst_key"))

raw_destinations = ids.select(F.col("Destination").alias("Source")).distinct()
fixed_destinations = ids.select(F.col("dst_key").alias("src_key")).distinct()

id_audit = {
    "source in scientific notation": ids.filter(U.is_scientific_notation(F.col("Source"))).count(),
    "source matches a destination (raw)": ids.join(raw_destinations, "Source", "left_semi").count(),
    "source matches a destination (repaired)": ids.join(fixed_destinations, "src_key", "left_semi").count(),
    "self-interactions after repair": ids.filter(F.col("src_key") == F.col("dst_key")).count(),
}
for name, value in id_audit.items():
    print(f"{name:<42}{value:>10,}  ({value / n_bronze_edges:.1%})")

bronze_twitter_graph.groupBy("Source Verified", "Destination Verified").count().show()

Verification flags are only `False` or null, so they are dropped; self-interactions (replies inside one's own thread) are kept but flagged.

In [0]:
typed_graph = (bronze_twitter_graph
    .withColumn("src", U.normalize_twitter_id(F.col("Source")))
    .withColumn("dst", U.normalize_twitter_id(F.col("Destination")))
    .filter(F.col("src").isNotNull() & F.col("dst").isNotNull())
    .withColumn("relationship_ts", F.to_timestamp("Relationship Date (UTC)"))
    .withColumn("src_joined", F.to_timestamp("Source Joined Date"))
    .withColumn("dst_joined", F.to_timestamp("Destination Joined Date")))

silver_edges = typed_graph.select(
    "src", "dst", F.col("Conversation ID").alias("conversation_id"), "relationship_ts",
    (F.col("src") == F.col("dst")).alias("is_self_interaction"))

src_side = typed_graph.select(F.col("src").alias("id"), "relationship_ts",
    F.col("Source Followers").alias("followers"), F.col("Source Followed").alias("following"),
    F.col("src_joined").alias("joined_ts"))
dst_side = typed_graph.select(F.col("dst").alias("id"), "relationship_ts",
    F.col("Destination Followers").alias("followers"), F.col("Destination Followed").alias("following"),
    F.col("dst_joined").alias("joined_ts"))
observations = src_side.unionByName(dst_side)

# Profile counts are snapshots taken at relationship time: keep the most recent one per account.
latest_profile = (observations.filter(F.col("followers").isNotNull())
    .groupBy("id").agg(
        F.max_by("followers", "relationship_ts").alias("followers"),
        F.max_by("following", "relationship_ts").alias("following"),
        F.min("joined_ts").alias("joined_ts")))

silver_vertices = (observations.groupBy("id")
    .agg(F.min("relationship_ts").alias("first_seen_ts"), F.max("relationship_ts").alias("last_seen_ts"))
    .join(latest_profile, "id", "left")
    .withColumn("has_profile", F.col("followers").isNotNull()))

U.write_table(silver_edges, "silver", "graph_edges")
U.write_table(silver_vertices, "silver", "graph_vertices")
silver_edges = U.read_table(spark, "silver", "graph_edges")
silver_vertices = U.read_table(spark, "silver", "graph_vertices")
print(f"silver edges: {silver_edges.count():,} | vertices: {silver_vertices.count():,}")
silver_vertices.show(5)

## 4. Gold: emotion profile, weighted social edges (no self-loops), account features and a data-quality log

In [0]:
gold_emotion_profile = (silver_mental_health.groupBy("Emotion", "label")
    .agg(F.count("*").alias("texts"),
         F.round(F.avg("word_count"), 1).alias("avg_words"),
         F.percentile_approx("word_count", 0.5).alias("median_words"),
         F.round(F.avg((F.col("word_count") > 200).cast("int")) * 100, 1).alias("pct_over_200_words"),
         F.round(F.avg((F.col("upper_ratio") > 0).cast("int")) * 100, 1).alias("pct_with_capitals"))
    .withColumn("share_pct", F.round(F.col("texts") / n_silver_text * 100, 1))
    .orderBy("label"))

U.write_table(gold_emotion_profile, "gold", "emotion_profile")
gold_emotion_profile.show()

In [0]:
gold_graph_edges = (silver_edges.filter(~F.col("is_self_interaction"))
    .groupBy("src", "dst")
    .agg(F.count("*").alias("weight"),
         F.countDistinct("conversation_id").alias("conversations"),
         F.min("relationship_ts").alias("first_ts"),
         F.max("relationship_ts").alias("last_ts")))

out_stats = gold_graph_edges.groupBy(F.col("src").alias("id")).agg(
    F.count("*").alias("out_degree"), F.sum("weight").alias("out_interactions"))
in_stats = gold_graph_edges.groupBy(F.col("dst").alias("id")).agg(
    F.count("*").alias("in_degree"), F.sum("weight").alias("in_interactions"))
self_stats = (silver_edges.filter("is_self_interaction")
    .groupBy(F.col("src").alias("id")).agg(F.count("*").alias("self_replies")))

gold_account_features = (silver_vertices
    .join(out_stats, "id", "left").join(in_stats, "id", "left").join(self_stats, "id", "left")
    .fillna(0, ["out_degree", "out_interactions", "in_degree", "in_interactions", "self_replies"])
    .withColumn("account_age_days", F.datediff("last_seen_ts", "joined_ts")))

U.write_table(gold_graph_edges, "gold", "graph_edges")
U.write_table(gold_account_features, "gold", "account_features")
U.read_table(spark, "gold", "account_features").orderBy(F.desc("in_interactions")).show(5)

In [0]:
quality_rows = [
    ("text", "bronze records", n_bronze_text),
    ("text", "silver records", n_silver_text),
    ("graph", "bronze relationships", n_bronze_edges),
    ("graph", "silver relationships (valid IDs)", silver_edges.count()),
    ("graph", "self-interactions", id_audit["self-interactions after repair"]),
    ("graph", "gold weighted edges", U.read_table(spark, "gold", "graph_edges").count()),
    ("graph", "accounts", silver_vertices.count()),
]
gold_data_quality = spark.createDataFrame(quality_rows, "dataset string, step string, rows long")
U.write_table(gold_data_quality, "gold", "data_quality")
gold_data_quality.show(truncate=False)

**Takeaway:** the ID repair turns a graph with 0% connected sources into one where ~56% of relationships link known accounts; ~32% of raw relationships are self-replies.